Animal classification by using a small crop of images, inspired by the past IOAI

comp link: https://www.kaggle.com/t/bf915ad970a148c39f666e0a6b6e0bdc

In [ ]:
import kagglehub

# Download latest version
root = kagglehub.competition_download('limitedcrop-animalclassification')

print("Path to competition files:", root)

Path to competition files: /root/.cache/kagglehub/competitions/limitedcrop-animalclassification


In [ ]:
from PIL import Image
import numpy as np
import pandas as pd
from glob import glob

paths = glob(root + '/test/*.jpg')
test_set = [Image.open(path) for path in paths]

In [ ]:
def generate_windows(img, window_size = 64, srtide = 32):
    windows = []
    for i in range(0, img.size[0], srtide):
        for j in range(0, img.size[1], srtide):
            window = img.crop((i, j, i + window_size, j + window_size))
            windows.append(window)
    return windows

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from transformers import CLIPProcessor, CLIPModel

MODEL_NAME = "openai/clip-vit-large-patch14"

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()  # we're only doing inference, no training
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded. Total parameters: {n_params/1e6:.1f}M")
print(f"Embedding dimension: {model.config.projection_dim}")

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Model loaded. Total parameters: 427.6M
Embedding dimension: 768


In [ ]:
@torch.no_grad()
def encode_images(pil_images):
    inputs = processor(images=list(pil_images), return_tensors="pt").to(device)
    feats = model.get_image_features(**inputs)
    # Newer transformers versions return an object; older return a tensor directly.
    if hasattr(feats, "image_embeds"):
        feats = feats.image_embeds
    elif hasattr(feats, "pooler_output"):
        feats = feats.pooler_output
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats

@torch.no_grad()
def encode_texts(texts):
    inputs = processor(text=list(texts), return_tensors="pt",
                       padding=True, truncation=True).to(device)
    feats = model.get_text_features(**inputs)
    if hasattr(feats, "text_embeds"):
        feats = feats.text_embeds
    elif hasattr(feats, "pooler_output"):
        feats = feats.pooler_output
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats

In [ ]:
prompt = encode_texts(["face"])

# prompt_vec = prompt.squeeze(0).detach().cpu().numpy()



In [ ]:
sims_per_image = []
embeddings_per_image = []

with torch.no_grad():
    for img in test_set:
        windows = generate_windows(img)

        prompt_vec = encode_images([img]).squeeze()
        feats = encode_images(windows)
        embeddings_per_image.append(feats)

        sims = np.zeros(len(windows))
        for i in range(len(windows)):
            sims[i] = np.dot(feats[i].detach().cpu().numpy(), prompt_vec.cpu().numpy())
        sims_per_image.append(sims)

best = np.argmax(sims_per_image, axis=1)

best_embeddings_list = []
for img_idx, best_window_idx in enumerate(best):
    best_embeddings_list.append(embeddings_per_image[img_idx][best_window_idx])

best_embeddings_all_images = torch.stack(best_embeddings_list)

In [ ]:
best_embeddings_all_images.shape

torch.Size([1000, 768])

In [ ]:
idx2class = pd.read_csv(root + '/classes.csv').to_dict()['class_name']

class_embds = encode_texts([f'a face of cats and dogs {cls}' for cls in idx2class.values()])

In [ ]:
class_embds.shape

torch.Size([20, 768])

In [ ]:
sims = best_embeddings_all_images @ class_embds.T
predictions = np.argmax(sims.detach().cpu().numpy(), axis=1)

In [ ]:
submission = pd.DataFrame()
submission['image_id'] = [path.split('/')[-1] for path in paths]
submission['prediction'] = [idx2class[pred] for pred in predictions]

submission.to_csv('submission.csv', index=False)

In [ ]:
submission

,image_id,prediction
0,img_0091.jpg,Persian
1,img_0728.jpg,Wheaten Terrier
2,img_0106.jpg,Birman
3,img_0711.jpg,Miniature Pinscher
4,img_0635.jpg,German Shorthaired
...,...,...
995,img_0592.jpg,Japanese Chin
996,img_0965.jpg,Abyssinian
997,img_0907.jpg,Japanese Chin
998,img_0615.jpg,Miniature Pinscher


In [ ]:
!kaggle competitions submit -c limitedcrop-animalclassification -f submission.csv -m "Your Message"

100% 26.8k/26.8k [00:00<00:00, 36.5kB/s]
Successfully submitted to animal classification with limited crop